# further classifications

In [ ]:
import os
from pathlib import Path
import pandas as pd
import duckdb as ddb
from connection_helper import sql
from pandas_plots import tbl, pls, hlp
from pandas_plots.hlp import add_bitmask_label
import duckdb as ddb

hlp.show_package_version(["pygwalker"])
os.environ['THEME']='light'
os.environ['DEBUG']='1'

dir_db=Path("C://temp") if hlp.get_os(hlp.OperatingSystem.WINDOWS) else Path(os.path.expanduser("~/tmp"))

file_db_clin = dir_db/'2026-02-26_data_clin.duckdb'
# file_db_clin = dir_db/'2025-11-11_data_clin.duckdb'
if not file_db_clin.exists():
    exit()

if not os.path.exists(".local"):
    os.makedirs(".local")

🐍 3.12.8 | 📦 pygwalker: 0.5.0.0 | 📦 pandas: 2.3.3 | 📦 numpy: 1.26.4 | 📦 duckdb: 1.4.3 | 📦 pandas-plots: 1.0.0 | 📦 connection-helper: 0.13.2


In [2]:
con = ddb.connect(":memory:")
# con = ddb.connect(file_db_clin, read_only=True)
_=con.execute("PRAGMA disable_progress_bar;")
_=con.execute(f"ATTACH DATABASE '{file_db_clin}' AS clin (READ_ONLY); SET SCHEMA 'clin';")
_=con.execute("ATTACH DATABASE ':memory:' AS mem;")


## <a id='toc1_1_'></a>[📆 data as of](#toc0_)

In [3]:
sql.print_meta(file_db_clin)

database file:           2025-11-11_data_clin.duckdb
data tag:                v2.3
last kkr data import:    2025-09-30
sql table created:       2025-11-11 11:52:01
doi:                     10.18444/5.03.01.0005.0021.0002
document created:        2026-02-27 20:05:17


In [4]:
# # * list all
print()
con.sql("""select distinct(Name) from Diagnose_WeitereKlassifikation order by Name""").to_df().Name.tolist()

['19q',
 '1p',
 '1p19q-Deletion',
 'AAIPI',
 'ABSTANDPT_MM',
 'ADOREG',
 'AEG',
 'AEG (Ösophagus)',
 'AEG n Siewert',
 'AEG nach Siewert',
 'AEG/Siewert (Ösophagus/Kardia)',
 'AEG/Siewert-Klassifikation',
 'AJC - Osteosarkome',
 'AJC - Weichteile',
 'AJC/UICC',
 'AJCC',
 'AJCC (Haut)',
 'AJCC (kutane T-Zell-Lymphome)',
 'AJCC (nicht Haut)',
 'AJCC 2016',
 'AJCC Klassifikation MM 2016',
 'AJCC Stadieneinteilung MM 2017',
 'AJCC+TNM (Malignes Melanom)',
 'AJCC-STADIUM',
 'AJCC-Stadieneinteilung',
 'AJCC-malignes Melanom',
 'ALK',
 'AML ELN - C92.0',
 'AML ELN-Klassifikation (2010)',
 'AML EuropLeukNet 2017',
 'AML EuropLeukNet 2017 (ELN)',
 'AML EuropLeukNet 2022',
 'AML European Leukemia Net',
 'AML European LeukemiaNet',
 'ANN_ARBOR',
 'ANN_ARBOR_BULKY',
 'ANN_ARBOR_EXTRA',
 'ANN_ARBOR_MILZ',
 'ANN_ARBOR_STADIUM',
 'ANN_ARBOR_ZUSATZ',
 'AP (alkalische Phosphatase)',
 'ASA Risikoklassifikation',
 'Adenokarzinome des ösophago-gastralen Übergangs (AEG), Klassifikation nach Siewer',
 'Allg

In [23]:
with open("../../sql/check_if_class.sql") as f:
    _macro_check_if_class = f.read()
_=con.execute(_macro_check_if_class)

In [24]:
db_class = (con.sql("""--sql
        with dia_fol as (
            select Name, Stadium, z_tum_id,
            'diag' as source 
            from Diagnose_WeitereKlassifikation
            union
            select Name, Stadium, z_tum_id,
            'fol' as source 
            from Folgeereignis_WeitereKlassifikation
        ),
        select
            tum.z_tum_id, Name, Stadium, z_kkr_label, source,
            mem.fx_check_if_class(Name) as class,
        from dia_fol
        join Tumor tum on dia_fol.z_tum_id = tum.z_tum_id
        where Stadium is not null
    """)
)
tbl.descr_db(db_class, "db_class")

        # t2 as (
        #     SELECT 
        #         tum.z_tum_id, t.Name, Stadium, z_kkr_label, source,
        #         --r.regex,
        #         case when regexp_matches(t.name::text, r.regex, 'i') then r.name end AS class,
        #     FROM dia_fol t
        #     join Tumor tum on t.z_tum_id = tum.z_tum_id
        #     JOIN '../../data/classification_regex.csv' r ON True --r.name = t.Name
        # )
        # select * from t2


# db_class.show(max_rows=50)

            # ,case
            #     when regexp_matches(Name, 'arbor', 'i') then 'ann_arbor'
            #     when regexp_matches(Name, 'uicc', 'i') then 'uicc'
            #     --when regexp_matches(Name, 'who|gehirn|brain', 'i') then 'brain' # 60k
            #     when regexp_matches(Name, 'tnm', 'i') then 'tnm'
            #     when regexp_matches(Name, 'psa', 'i') then 'psa'
            #     when regexp_matches(Name, 'breslow', 'i') then 'breslow'
            #     when regexp_matches(Name, 'clark', 'i') then 'clark'
            #     when regexp_matches(Name, 'gleason', 'i') then 'gleason'
            #     when regexp_matches(Name, 'gehirn|brain|zns|gliom|id-h|1p19q|astrocytoma|glioblastoma|meningioma|koos|knosp|who.*(gehirn|brain|zns|hirn)', 'i') then 'brain'
            #     when regexp_matches(Name, 'p16|hpv|pap', 'i') then 'hpv'
            #     when regexp_matches(Name, 'epstein|isup|grade group', 'i') then 'isup_grade_group'
            #     when regexp_matches(Name, 'hep', 'i') then 'hep' -- M+(HEP) for colorectal cancer
            #     when regexp_matches(Name, 'ki67|ki-67', 'i') then 'ki67'
            #     -- 1. Hormone Receptor Label (ER/PR)
            #     when regexp_matches(Name, 'pgr|strogen|rezeptor|receptor|er %', 'i') then 'hormone_receptor'
            #     -- 2. HER2 Label
            #     when regexp_matches(Name, 'her-2|her2|erbb2', 'i') then 'her2'

            #     when regexp_matches(Name, 'siewert|aeg', 'i') then 'siewert_aeg_type'
            #     when regexp_matches(Name, 'dukes', 'i') then 'dukes_stage'
            #     when regexp_matches(Name, 'lauren', 'i') then 'lauren_histotype'

            #     when regexp_matches(Name, 'binet|rai', 'i') then 'cll_staging'
            #     when regexp_matches(Name, 'durie|salmon|iss|myeloma', 'i') then 'myeloma_staging'
            #     when regexp_matches(Name, 'eln|fab|leukämie|leukemia', 'i') then 'leukemia_grading'
            #     when regexp_matches(Name, 'who', 'i') then 'who'
            # end as class

ParserException: Parser Error: syntax error at or near "select"

LINE 11:         select
                 ^

In [19]:
if os.getenv("DEBUG") == "1":
    db_class.filter("class = 'her2'").unique("Name").show()
    db_class.filter("class = 'her2'").show(max_rows=50)

┌──────────────────────────────────┐
│               Name               │
│             varchar              │
├──────────────────────────────────┤
│ Her2-neu Status                  │
│ HER2-NEU                         │
│ HER2-neu (C00-C14,C15,C16)       │
│ Her2neu                          │
│ HER2/NEU (C00-C14,C15,C16) [P/N] │
│ HER2-neu-Überexpression          │
│ HER2_Status                      │
│ Her2neuStatus                    │
│ HER-2-neu                        │
│ HER2NEU                          │
│ HER2-neu                         │
├──────────────────────────────────┤
│             11 rows              │
└──────────────────────────────────┘

┌──────────────────────────────────────┬──────────┬─────────┬─────────────┬─────────┬─────────┐
│               z_tum_id               │   Name   │ Stadium │ z_kkr_label │ source  │  class  │
│               varchar                │ varchar  │ varchar │   varchar   │ varchar │ varchar │
├──────────────────────────────────────┼─────

In [20]:
db_class_agg = (db_class
    .aggregate("z_kkr_label, source, class, count(*) as cnt, count(distinct z_tum_id) as cnt_tum")
)

In [21]:
_df = db_class.aggregate("class, source, count(*) as cnt").to_df()
pls.plot_stacked_bars(
    _df,
    orientation="h",
    sort_values_index=True,
    height=800,
    normalize=True,
    show_pct_all=True,
)

In [ ]:
# _df = (
#     db_class_agg
#     # .project("class, source, z_kkr_label")
#     .project("class, source, z_kkr_label, cnt")
#     .to_df()
#     .dropna()
# )
# _ = pls.plot_facet_stacked_bars(
#     _df,
#     renderer=None,
#     # relative=True,
#     annotations=True,
# )

In [ ]:
_df = db_class_agg.project("z_kkr_label, class, cnt").to_df().dropna()
tbl.pivot_df(
    _df,
    swap=True,
    data_bar_axis="",
    heatmap_axis="xy",
    pct_axis="",
    total_axis="xy",
    total_exclude=True,
)


z_kkr_label,01-SH,02-HH,03-NI,04-HB,05-NW,06-HE,07-RP,08-BW,09-BY,10-SL,11-BE,12-BB,13-MV,14-SN,15-ST,16-TH,Total
class,,,,,,,,,,,,,,,,,
ann_arbor,2_054,968,2_400,425,0,6_529,1_215,8_600,8_464,465,6_467,4_567,1_985,5_808,2_092,1_526,53_565
brain,323,0,1_784,457,0,3_308,679,10_813,6_765,701,2_634,2_030,65,4_332,2_115,1_285,37_291
breslow,37,442,700,639,0,1_668,188,3_403,973,700,2_354,1_029,300,197,3_469,1_261,17_360
clark,410,292,1_157,199,0,1_627,1_647,5_070,3_743,999,3_853,3_571,1_294,4_569,3_178,2_307,33_916
cll_staging,300,90,610,31,0,645,287,1_155,1_719,122,446,737,752,2_040,1_119,523,10_576
dukes_stage,0,0,0,0,0,1,0,169,154,0,0,2,20,4,355,0,705
gleason,41,334,730,304,0,5_411,805,15_949,9_773,1_580,806,1_627,1_451,5_623,3_468,1_704,49_606
her2,141,596,445,207,6_641,0,181,5_709,303,1_052,22,16,455,0,0,14,15_782
hormone_receptor,2,0,42,12,13_918,0,12,3_919,10,0,0,0,0,0,0,22,17_937


In [ ]:
if False:
    import pygwalker as pyg

    pyg.walk(
        db_class_agg.to_df().dropna(),
        kernel_computation=True,
    )